In [1]:
import pandas as pd
import sqlite3
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

# Load all clean data
data = load_all_data()

pl        = data['profitandloss']
bs        = data['balancesheet']
cf        = data['cashflow']
companies = data['companies']

print("Data loaded and ready for validation!")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Data loaded and ready for validation!


In [2]:
# This list will collect all rule violations
violations = []

def log_violation(rule_id, rule_name, company_id, year, field, issue, severity):
    """Logs a data quality violation to our tracker."""
    violations.append({
        'rule_id':    rule_id,
        'rule_name':  rule_name,
        'company_id': company_id,
        'year':       year,
        'field':      field,
        'issue':      issue,
        'severity':   severity
    })

print("Violation tracker ready!")

Violation tracker ready!


In [3]:
# ── DQ-01: Company PK Uniqueness ─────────────────────────────────────────────
print("Running DQ-01: Company PK Uniqueness...")
duplicate_companies = companies[companies['id'].duplicated()]
if len(duplicate_companies) > 0:
    for _, row in duplicate_companies.iterrows():
        log_violation('DQ-01', 'Company PK Uniqueness',
                      row['id'], None, 'id',
                      'Duplicate company ticker found', 'CRITICAL')
    print(f"  ❌ {len(duplicate_companies)} duplicate companies found!")
else:
    print(f"  ✅ All {len(companies)} company IDs are unique")

# ── DQ-02: Annual PK Uniqueness ──────────────────────────────────────────────
print("\nRunning DQ-02: Annual PK Uniqueness...")
for table_name, df in [('profitandloss', pl), ('balancesheet', bs), ('cashflow', cf)]:
    dupes = df[df.duplicated(subset=['company_id', 'year'])]
    if len(dupes) > 0:
        print(f"  ❌ {table_name}: {len(dupes)} duplicate (company_id, year) pairs")
        for _, row in dupes.iterrows():
            log_violation('DQ-02', 'Annual PK Uniqueness',
                          row['company_id'], row['year'], 'company_id+year',
                          f'Duplicate row in {table_name}', 'CRITICAL')
    else:
        print(f"  ✅ {table_name}: No duplicates found")

Running DQ-01: Company PK Uniqueness...
  ✅ All 92 company IDs are unique

Running DQ-02: Annual PK Uniqueness...
  ✅ profitandloss: No duplicates found
  ✅ balancesheet: No duplicates found
  ✅ cashflow: No duplicates found


In [4]:
# ── DQ-03: FK Integrity ───────────────────────────────────────────────────────
print("Running DQ-03: Foreign Key Integrity...")

valid_ids = set(companies['id'].unique())

for table_name, df in [('profitandloss', pl), ('balancesheet', bs), ('cashflow', cf)]:
    orphans = df[~df['company_id'].isin(valid_ids)]
    if len(orphans) > 0:
        print(f"  ❌ {table_name}: {len(orphans)} orphan rows found")
        for _, row in orphans.iterrows():
            log_violation('DQ-03', 'FK Integrity',
                          row['company_id'], row.get('year'), 'company_id',
                          f'company_id not found in companies table', 'CRITICAL')
    else:
        print(f"  ✅ {table_name}: All company IDs valid")

Running DQ-03: Foreign Key Integrity...
  ❌ profitandloss: 91 orphan rows found
  ❌ balancesheet: 25 orphan rows found
  ❌ cashflow: 96 orphan rows found


In [5]:
# ── DQ-04: Balance Sheet Balance ─────────────────────────────────────────────
print("Running DQ-04: Balance Sheet Balance...")
bs_check = bs.copy()
bs_check['diff_pct'] = abs(
    bs_check['total_assets'] - bs_check['total_liabilities']
) / bs_check['total_assets'].replace(0, float('nan')) * 100

unbalanced = bs_check[bs_check['diff_pct'] > 1.0]
if len(unbalanced) > 0:
    print(f"  ⚠️  {len(unbalanced)} rows where assets ≠ liabilities (>1% diff)")
    for _, row in unbalanced.iterrows():
        log_violation('DQ-04', 'Balance Sheet Balance',
                      row['company_id'], row['year'], 'total_assets',
                      f"Assets vs Liabilities diff: {row['diff_pct']:.2f}%", 'WARNING')
else:
    print("  ✅ All balance sheets balanced")

# ── DQ-05: OPM Cross Check ───────────────────────────────────────────────────
print("\nRunning DQ-05: OPM Cross Check...")
pl_check = pl.copy()
pl_check['computed_opm'] = (
    pl_check['operating_profit'] / pl_check['sales'].replace(0, float('nan')) * 100
)
pl_check['opm_diff'] = abs(pl_check['opm_percentage'] - pl_check['computed_opm'])

opm_issues = pl_check[pl_check['opm_diff'] > 1.0]
if len(opm_issues) > 0:
    print(f"  ⚠️  {len(opm_issues)} rows where OPM doesn't match computed value")
    for _, row in opm_issues.iterrows():
        log_violation('DQ-05', 'OPM Cross Check',
                      row['company_id'], row['year'], 'opm_percentage',
                      f"OPM diff: {row['opm_diff']:.2f}%", 'WARNING')
else:
    print("  ✅ All OPM values match computed values")

Running DQ-04: Balance Sheet Balance...
  ✅ All balance sheets balanced

Running DQ-05: OPM Cross Check...
  ⚠️  216 rows where OPM doesn't match computed value


In [6]:
# ── DQ-06: Positive Sales ────────────────────────────────────────────────────
print("Running DQ-06: Positive Sales...")
zero_sales = pl[pl['sales'] <= 0]
if len(zero_sales) > 0:
    print(f"  ⚠️  {len(zero_sales)} rows with zero or negative sales")
    for _, row in zero_sales.iterrows():
        log_violation('DQ-06', 'Positive Sales',
                      row['company_id'], row['year'], 'sales',
                      f"Sales value: {row['sales']}", 'WARNING')
else:
    print("  ✅ All sales values are positive")

# ── DQ-07: Year Format ───────────────────────────────────────────────────────
print("\nRunning DQ-07: Year Format...")
import re
bad_years = pl[~pl['year'].str.match(r'^\d{4}-\d{2}$', na=False)]
if len(bad_years) > 0:
    print(f"  ❌ {len(bad_years)} rows with invalid year format")
    for _, row in bad_years.iterrows():
        log_violation('DQ-07', 'Year Format',
                      row['company_id'], row['year'], 'year',
                      f"Invalid year format: {row['year']}", 'CRITICAL')
else:
    print("  ✅ All year formats are valid")

# ── DQ-08: Ticker Format ─────────────────────────────────────────────────────
print("\nRunning DQ-08: Ticker Format...")
bad_tickers = companies[
    (companies['id'].str.len() < 2) | (companies['id'].str.len() > 12)
]
if len(bad_tickers) > 0:
    print(f"  ⚠️  {len(bad_tickers)} tickers with invalid length")
    for _, row in bad_tickers.iterrows():
        log_violation('DQ-08', 'Ticker Format',
                      row['id'], None, 'id',
                      f"Ticker length out of range: {row['id']}", 'CRITICAL')
else:
    print("  ✅ All ticker formats are valid")

Running DQ-06: Positive Sales...
  ⚠️  1 rows with zero or negative sales

Running DQ-07: Year Format...
  ✅ All year formats are valid

Running DQ-08: Ticker Format...
  ✅ All ticker formats are valid


In [7]:
# ── DQ-09: Net Cash Check ────────────────────────────────────────────────────
print("Running DQ-09: Net Cash Check...")
cf_check = cf.copy()
cf_check['computed_net'] = (
    cf_check['operating_activity'] +
    cf_check['investing_activity'] +
    cf_check['financing_activity']
)
cf_check['cash_diff'] = abs(cf_check['net_cash_flow'] - cf_check['computed_net'])
cash_issues = cf_check[cf_check['cash_diff'] > 10]
print(f"  ⚠️  {len(cash_issues)} rows where net cash flow doesn't match sum")
for _, row in cash_issues.iterrows():
    log_violation('DQ-09', 'Net Cash Check',
                  row['company_id'], row['year'], 'net_cash_flow',
                  f"Diff: {row['cash_diff']:.0f} Cr", 'WARNING')

# ── DQ-10: Non-Negative Fixed Assets ─────────────────────────────────────────
print("\nRunning DQ-10: Non-Negative Fixed Assets...")
neg_assets = bs[bs['fixed_assets'] < 0]
if len(neg_assets) > 0:
    print(f"  ⚠️  {len(neg_assets)} rows with negative fixed assets")
    for _, row in neg_assets.iterrows():
        log_violation('DQ-10', 'Non-Negative Fixed Assets',
                      row['company_id'], row['year'], 'fixed_assets',
                      f"Fixed assets: {row['fixed_assets']}", 'WARNING')
else:
    print("  ✅ All fixed asset values are non-negative")

# ── DQ-11: Tax Rate Range ────────────────────────────────────────────────────
print("\nRunning DQ-11: Tax Rate Range...")
bad_tax = pl[(pl['tax_percentage'] < 0) | (pl['tax_percentage'] > 60)]
if len(bad_tax) > 0:
    print(f"  ⚠️  {len(bad_tax)} rows with unusual tax rate")
    for _, row in bad_tax.iterrows():
        log_violation('DQ-11', 'Tax Rate Range',
                      row['company_id'], row['year'], 'tax_percentage',
                      f"Tax rate: {row['tax_percentage']}%", 'WARNING')
else:
    print("  ✅ All tax rates are within range")

# ── DQ-12: Dividend Payout Cap ───────────────────────────────────────────────
print("\nRunning DQ-12: Dividend Payout Cap...")
high_div = pl[pl['dividend_payout'] > 200]
if len(high_div) > 0:
    print(f"  ⚠️  {len(high_div)} rows with unusually high dividend payout")
    for _, row in high_div.iterrows():
        log_violation('DQ-12', 'Dividend Payout Cap',
                      row['company_id'], row['year'], 'dividend_payout',
                      f"Payout: {row['dividend_payout']}%", 'WARNING')
else:
    print("  ✅ All dividend payout values are within range")

# ── DQ-14: EPS Sign Consistency ──────────────────────────────────────────────
print("\nRunning DQ-14: EPS Sign Consistency...")
eps_issues = pl[(pl['net_profit'] > 0) & (pl['eps'] <= 0)]
if len(eps_issues) > 0:
    print(f"  ⚠️  {len(eps_issues)} rows where profit is positive but EPS is not")
    for _, row in eps_issues.iterrows():
        log_violation('DQ-14', 'EPS Sign Consistency',
                      row['company_id'], row['year'], 'eps',
                      f"Net profit: {row['net_profit']}, EPS: {row['eps']}", 'WARNING')
else:
    print("  ✅ All EPS values consistent with net profit")

# ── DQ-16: Coverage Check ────────────────────────────────────────────────────
print("\nRunning DQ-16: Coverage Check...")
coverage = pl.groupby('company_id')['year'].count()
low_coverage = coverage[coverage < 5]
if len(low_coverage) > 0:
    print(f"  ⚠️  {len(low_coverage)} companies with less than 5 years of data")
    for company_id, count in low_coverage.items():
        log_violation('DQ-16', 'Coverage Check',
                      company_id, None, 'year',
                      f"Only {count} years of data available", 'WARNING')
else:
    print("  ✅ All companies have 5+ years of data")

Running DQ-09: Net Cash Check...
  ⚠️  1 rows where net cash flow doesn't match sum

Running DQ-10: Non-Negative Fixed Assets...
  ✅ All fixed asset values are non-negative

Running DQ-11: Tax Rate Range...
  ⚠️  108 rows with unusual tax rate

Running DQ-12: Dividend Payout Cap...
  ⚠️  7 rows with unusually high dividend payout

Running DQ-14: EPS Sign Consistency...
  ⚠️  5 rows where profit is positive but EPS is not

Running DQ-16: Coverage Check...
  ⚠️  1 companies with less than 5 years of data


In [8]:
# Save all violations to CSV
violations_df = pd.DataFrame(violations)

if len(violations_df) > 0:
    violations_df.to_csv('output/validation_failures.csv', index=False)
    print(f"validation_failures.csv saved with {len(violations_df)} violations!")
    print()
    print("Summary by severity:")
    print(violations_df['severity'].value_counts().to_string())
    print()
    print("Summary by rule:")
    print(violations_df['rule_id'].value_counts().to_string())
else:
    pd.DataFrame(columns=['rule_id','rule_name','company_id',
                           'year','field','issue','severity']
    ).to_csv('output/validation_failures.csv', index=False)
    print("✅ Zero violations found! Clean data!")

validation_failures.csv saved with 551 violations!

Summary by severity:
severity
WARNING     339
CRITICAL    212

Summary by rule:
rule_id
DQ-05    216
DQ-03    212
DQ-11    108
DQ-12      7
DQ-14      5
DQ-06      1
DQ-09      1
DQ-16      1


In [9]:
# Check which company IDs are causing DQ-03 violations
valid_ids = set(companies['id'].unique())

orphans_pl = pl[~pl['company_id'].isin(valid_ids)]
print("Company IDs in P&L but NOT in companies table:")
print(orphans_pl['company_id'].unique())
print(f"\nTotal: {len(orphans_pl['company_id'].unique())} companies")

Company IDs in P&L but NOT in companies table:
['ULTRACEMCO' 'UNIONBANK' 'UNITDSPR' 'VBL' 'VEDL' 'WIPRO' 'ZOMATO'
 'ZYDUSLIFE']

Total: 8 companies


In [10]:
# Document this finding clearly
print("DQ-03 FINDING:")
print("=" * 50)
print("8 companies exist in financial statements")
print("but are missing from companies master table.")
print()
print("These are valid Nifty 100 companies.")
print("This is a source data gap, not a code error.")
print()
print("Impact:")
print("  - These companies load correctly into P&L,")
print("    Balance Sheet and Cash Flow tables")
print("  - They will be excluded from any analysis")
print("    that requires a JOIN with companies table")
print("  - Total affected rows:", len(pl[~pl['company_id'].isin(valid_ids)]))

DQ-03 FINDING:
8 companies exist in financial statements
but are missing from companies master table.

These are valid Nifty 100 companies.
This is a source data gap, not a code error.

Impact:
  - These companies load correctly into P&L,
    Balance Sheet and Cash Flow tables
  - They will be excluded from any analysis
    that requires a JOIN with companies table
  - Total affected rows: 91


In [11]:
import os

print("=" * 50)
print("SPRINT 1 — FINAL DELIVERABLES CHECK")
print("=" * 50)

files_to_check = [
    # Python modules
    'src/__init__.py',
    'src/etl/__init__.py',
    'src/etl/loader.py',
    'src/etl/database.py',
    'src/etl/validator.py',
    # Database
    'data/nifty100.db',
    # Output files
    'output/load_audit.csv',
    'output/validation_failures.csv',
    # Notebooks
    'notebooks/01_data_exploration.ipynb',
    'notebooks/02_data_cleaning.ipynb',
    'notebooks/03_database.ipynb',
    'notebooks/04_data_quality.ipynb',
]

all_good = True
for f in files_to_check:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {f}")
    if not exists:
        all_good = False

print()
if all_good:
    print("All deliverables present! Ready to submit! 🚀")
else:
    print("Some files are missing! Fix before submitting!")

SPRINT 1 — FINAL DELIVERABLES CHECK
  ✅  src/__init__.py
  ✅  src/etl/__init__.py
  ✅  src/etl/loader.py
  ✅  src/etl/database.py
  ✅  src/etl/validator.py
  ✅  data/nifty100.db
  ✅  output/load_audit.csv
  ✅  output/validation_failures.csv
  ✅  notebooks/01_data_exploration.ipynb
  ✅  notebooks/02_data_cleaning.ipynb
  ✅  notebooks/03_database.ipynb
  ✅  notebooks/04_data_quality.ipynb

All deliverables present! Ready to submit! 🚀
